<a href="https://colab.research.google.com/github/Profilo0/IDV_project/blob/main/IDV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install jupyter_dash dash --quiet
!pip install portpicker --quiet
import portpicker
import jupyter_dash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.4 MB/s eta 0:00:00


In [3]:
"""
dashboard_v3.py  —  Padova ↔ Helsinki  (v3: Decision-Support Redesign)
=======================================================================
Grounded in Oral et al. (2023) "From Information to Choice":
  Intelligence stage → Salary, Prices, Wages, City Quality, Datasets tabs
  Design stage       → "What If?" tab: compare actual Finnish offer vs Italian salary
  Choice stage       → "My Decision" tab: preference elicitation + weighted verdict

Accessibility changes:
  • Light theme throughout (WCAG AA contrast ≥ 4.5:1)
  • Tooltips: white background, dark text, readable at any size
  • Minimum 12px font on all labels; 13px on body text
  • Colour encodes meaning, not decoration; annotations replace reliance on colour alone

Dash 4.x: uses jupyter_mode= (not mode=)
"""

import warnings; warnings.filterwarnings("ignore")
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import dash

import dash
from dash import dcc, html, Input, Output, State, callback_context, ALL

port = portpicker.pick_unused_port()

# ── JupyterDash for Colab compatibility ──
IN_COLAB = "google.colab" in str(globals().get("__builtins__", ""))

import threading
from google.colab import output


from interaction_log import log_stores, log_panel, register_callbacks

# ─── ENVIRONMENT ──────────────────────────────────────────────────────────────
DATA_DIRS = [Path("/content"), Path(".")]

def find_file(name):
    for d in DATA_DIRS:
        p = d / name
        if p.exists(): return p
    return None

# ─── LIGHT COLOUR PALETTE ─────────────────────────────────────────────────────
# Designed for WCAG AA contrast on white/near-white backgrounds
C = dict(
    bg    ="#F7F5F0",   # warm off-white page background
    surf  ="#FFFFFF",   # card / chart surface
    panel ="#F0EDE8",   # slightly warm panel
    border="#D6D1C8",   # subtle border
    pad   ="#B85C28",   # Padova terracotta — darkened for contrast on white (5.2:1)
    hel   ="#1E6FAD",   # Helsinki blue — darkened (5.1:1)
    gold  ="#7A5F0A",   # gold/amber — darkened (5.4:1)
    good  ="#166534",   # green — darkened (6.1:1)
    warn  ="#B91C1C",   # red/warn — darkened (5.9:1)
    text  ="#1C1917",   # near-black (15:1 on white)
    muted ="#57534E",   # medium grey (5.5:1 on white)
    tip_bg="#FFFFFF",   # tooltip background
    tip_bd="#D6D1C8",   # tooltip border
)
PLI          = 1.221
RECENT_CUTOFF= 2014

# Plotly tooltip style — white background, dark text, always readable
TIP = dict(bgcolor=C["tip_bg"], bordercolor=C["tip_bd"],
           font=dict(size=12, color=C["text"]))

LAYOUT = dict(
    paper_bgcolor=C["bg"], plot_bgcolor=C["surf"],
    font=dict(family="IBM Plex Sans, sans-serif", color=C["text"], size=12),
    margin=dict(l=70, r=40, t=48, b=55),
    xaxis=dict(gridcolor=C["border"], zeroline=False,
               linecolor=C["border"], tickfont=dict(size=11)),
    yaxis=dict(gridcolor=C["border"], zeroline=False,
               linecolor=C["border"], tickfont=dict(size=11)),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(size=12),
                bordercolor=C["border"], borderwidth=1),
    hoverlabel=TIP,
)

def LO(**overrides):
    base = {k:v for k,v in LAYOUT.items() if k not in overrides}
    return {**base, **overrides}

# ─── LOAD DATA ────────────────────────────────────────────────────────────────
def load(name, req=True):
    p = find_file(name)
    if p:
        df = pd.read_csv(p)
        print(f"  ✓ {name:52s} {len(df):>7} rows")
        return df
    if req: print(f"  ✗ {name:52s} NOT FOUND")
    return pd.DataFrame()

print("Loading datasets …")
sal    = load("tableau_salary_range.csv")
be     = load("tableau_breakeven.csv")
hicp   = load("tableau_hicp_categories.csv")
wages  = load("tableau_fi_wage_distribution.csv")
ua_ref = load("tableau_urban_audit_indicators.csv")
nmb    = load("numbeo_comparison.csv")
e_lc   = load("estat_urb_living_conditions.csv",                req=False)
e_tr   = load("estat_urb_transport.csv",               req=False)
e_inc  = load("estat_ilc_di03_income_nuts2.csv",       req=False)
e_emp  = load("estat_lfst_employment_rates.csv",       req=False)
e_hou  = load("estat_ilc_lvho01_housing_overburden.csv",req=False)
e_hou2 = load("estat_ilc_lvho07a_housing_cost_share.csv",req=False)



if not wages.empty:
    wages = wages.drop_duplicates(subset=["occ_label"]).reset_index(drop=True)

OCC_LABELS  = wages["occ_label"].tolist() if not wages.empty else ["All occupations"]
NUMBEO_CATS = (["All"] + sorted(nmb["category"].unique().tolist())
               if not nmb.empty else ["All"])

# ─── URBAN AUDIT PARSING ──────────────────────────────────────────────────────
UA_INDICATOR_MAP = {
    "EC1001I": ("Employment rate 20–64", "%",         True),
    "EC2001I": ("Unemployment rate",     "%",         False),
    "ED1001I": ("Tertiary education",    "%",         True),
    "LS1001I": ("Life satisfaction",     "score 0–10",True),
    "HA2001I": ("Housing cost / income", "%",         False),
    "SA1002I": ("Feel safe at night",    "% agree",   True),
    "EN1012I": ("Air quality PM2.5",     "µg/m³",     False),
    "SA3002I": ("Homicide rate",         "per 100k",  False),
    "GR1001I": ("Green space access",    "%",         True),
    "TT1005I": ("Satisfaction w/ transport","% satisfied",True),
    "TR1007I": ("Public transport use",  "% of trips",True),
    "TR1010I": ("Average commute time",  "minutes",   False),
    "EC4001V": ("GDP per capita",        "€ PPS",     True),
}

def parse_ua():
    frames = []
    for df in [e_lc, e_tr]:
        if df.empty: continue
        gc = next((c for c in df.columns if c.lower() in ("cities","geo","ref_area")), None)
        ic = next((c for c in df.columns if c.lower() in ("indic_ur","indicator","indic")), None)
        vc = next((c for c in df.columns if c.lower() in ("obs_value","value","values")), None)
        tc = next((c for c in df.columns if c.lower() in ("time","year","time_period")), None)
        if not (gc and ic and vc): continue
        sub = df[df[gc].isin({"IT004C","FI001C"})].copy()
        sub[vc] = pd.to_numeric(sub[vc], errors="coerce")
        sub = sub.dropna(subset=[vc])
        if tc:
            sub[tc] = sub[tc].astype(str).str[:4]
            sub = sub[pd.to_numeric(sub[tc], errors="coerce") >= RECENT_CUTOFF]
            sub = sub.sort_values(tc).groupby([gc, ic]).last().reset_index()
        frames.append(sub.rename(columns={gc:"city_code", ic:"indic", vc:"value"}))
    if frames:
        all_ua = pd.concat(frames, ignore_index=True)
        rows = []
        for code, (name, unit, hi_better) in UA_INDICATOR_MAP.items():
            sub = all_ua[all_ua["indic"] == code]
            pad = sub[sub["city_code"]=="IT004C"]["value"].mean()
            hel = sub[sub["city_code"]=="FI001C"]["value"].mean()
            if pd.isna(pad) or pd.isna(hel): continue
            diff = round((hel-pad)/abs(pad)*100,1) if pad!=0 else 0
            favours = "Helsinki" if (hel<pad if not hi_better else hel>pad) else "Padova"
            rows.append(dict(indicator=name, padova=round(pad,2), helsinki=round(hel,2),
                             unit=unit, favours=favours, diff_pct=diff,
                             higher_is_better=hi_better))
        if rows:
            print(f"  ✓ Real Urban Audit: {len(rows)} indicators parsed")
            return pd.DataFrame(rows)
    if not ua_ref.empty:
        if "higher_is_better" not in ua_ref.columns:
            ua_ref["higher_is_better"] = ua_ref.get("favours","Helsinki").apply(
                lambda f: True)
        print("  → Using reference Urban Audit values")
        return ua_ref
    return pd.DataFrame()

ua = parse_ua()

# ─── CRITERIA SCORES FOR CHOICE STAGE ────────────────────────────────────────
# Normalised 0–10 scores (10=best) for 8 criteria; Padova and Helsinki.
# Derived from collected data; salary score is dynamic (computed from salary slider).
# Sources documented in tooltip of each slider.
CRITERIA = [
    # (id,  label,          pad_score, hel_score, data_source)
    ("qol",  "Quality of life",          7.2,  7.8, "Urban Audit life satisfaction (0–10 scale)"),
    ("safe", "Safety",                   7.3,  8.2, "Urban Audit % feel safe walking at night"),
    ("air",  "Air quality",              3.5,  8.5, "Eurostat PM2.5 µg/m³ (inverted: lower=better score)"),
    ("trans","Public transport",         5.7,  7.0, "Urban Audit % satisfied with public transport"),
    ("hous", "Housing affordability",    6.5,  5.0, "Eurostat housing cost as % of disposable income (inverted)"),
    ("col",  "Cost of living",           7.5,  5.0, "Eurostat HICP price level index (FI=122 vs IT=100, inverted)"),
    ("emp",  "Employment opportunities", 6.8,  7.9, "Eurostat employment rate 20–64 (%)"),
    ("edu",  "Education & innovation",   6.2,  8.1, "Urban Audit tertiary education attainment (%)"),
    # salary score is added dynamically in the Choice callback
]

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def compute_criteria_scores(ua):
    """Map Urban Audit indicators to 0-10 scores for Padova and Helsinki."""
    if ua is None or ua.empty:
        return None
    # mapping of criterion id -> indicator name in ua
    indicator_map = {
        "qol":   "Life satisfaction",
        "safe":  "Feel safe at night",
        "air":   "Air quality PM2.5",
        "trans": "Satisfaction w/ transport",
        "hous":  "Housing cost / income",
        "emp":   "Employment rate 20–64",
        "edu":   "Tertiary education",
        # "col" will be handled separately using HICP, see note below
    }
    scores = {}
    for cid, ind_name in indicator_map.items():
        row = ua[ua["indicator"] == ind_name]
        if row.empty:
            continue
        is_higher_better = row["higher_is_better"].iloc[0]
        pad_val = row["padova"].iloc[0]
        hel_val = row["helsinki"].iloc[0]
        # Normalize to 0-10: for higher-better, score = (value/10)*10 = value if max 10.
        # We need a sensible scale. You can use the maximum observed across both cities.
        # Here we assume the indicator's natural max (e.g., % satisfaction out of 100 -> divide by 10)
        # For simplicity, scale between 0 and 10 using min-max across the two cities.
        # This is a basic approach; adjust depending on indicator units.
        if is_higher_better:
            # higher is better: 10 = max observed
            max_val = max(pad_val, hel_val)
            scores[cid] = {
                "padova": round((pad_val / max_val) * 10, 1) if max_val != 0 else 0,
                "helsinki": round((hel_val / max_val) * 10, 1) if max_val != 0 else 0
            }
        else:
            # lower is better: 10 = min observed
            min_val = min(pad_val, hel_val)
            scores[cid] = {
                "padova": round((min_val / pad_val) * 10, 1) if pad_val != 0 else 0,
                "helsinki": round((min_val / hel_val) * 10, 1) if hel_val != 0 else 0
            }
    # Add cost of living (col) using HICP index if available
    # We'll use a flag: assume FI HICP = 122, IT = 100, higher is worse.
    # Score = 10 - (index/12.2) or similar.
    # This part can be refined; for now set placeholder.
    scores["col"] = {"padova": 7.5, "helsinki": 5.0}  # keep your existing values or compute from HICP

    return scores

def nearest_row(df, col, val):
    idx = (df[col] - val).abs().idxmin()
    return df.loc[idx]

def tax_row(gross):
    if sal.empty: return None
    return nearest_row(sal, "gross_annual", gross)

def get_kpis(gross, exp_pad, exp_hel):
    row = tax_row(gross)
    if row is None: return ["—"]*5
    ni, nf = row.net_monthly_italy, row.net_monthly_finland
    pp = row.net_monthly_fi_ppp_adj
    disp = (nf - exp_hel) - (ni - exp_pad)
    bev = nearest_row(be, "gross_italy", gross).fi_gross_ppp_parity if not be.empty else gross*1.28
    return [f"€{ni:,.0f}/mo", f"€{nf:,.0f}/mo", f"€{pp:,.0f}/mo", f"€{bev:,.0f}",
            ((f"+€{disp:,.0f}/mo" if disp>=0 else f"−€{abs(disp):,.0f}/mo"), disp)]

ACTIVITIES = [
    ("coffee",   "☕ Espresso",            "Daily",   1,   "Cappuccino"),
    ("lunch_r",  "🍽  Lunch (restaurant)",  "Daily",   1,   "Meal at an Inexpensive"),
    ("dinner",   "🍷 Dinner for 2",         "Weekly",  1,   "Meal for Two"),
    ("beer",     "🍺 Beer at bar",           "Weekly",  2,   "Domestic Draft Beer"),
    ("transport","🚌 Transit pass",          "Monthly", 1,   "Monthly Pass"),
    ("gym",      "💪 Gym membership",        "Monthly", 1,   "Fitness Club"),
    ("cinema",   "🎬 Cinema",               "Monthly", 2,   "Cinema"),
    ("wine",     "🍾 Bottle of wine",        "Weekly",  1,   "Bottle of Wine"),
    ("internet", "📡 Internet/mobile",       "Monthly", 1,   "Mobile Phone Plan"),
    ("utilities","💡 Basic utilities",       "Monthly", 1,   "Basic utilities"),
    ("gas",      "⛽ Petrol (50L)",          "Monthly", 2,   "Gasoline"),
    ("rent_1br", "🏠 Rent 1-BR apartment",   "Monthly", 1,   "1 Bedroom Apartment"),
]
FREQ_MULT = {"Daily":21.67,"Weekly":4.33,"Monthly":1.0}

def nmb_price(match, col):
    if nmb.empty: return None
    m = nmb[nmb["item"].str.contains(match, case=False, na=False)]
    return float(m.iloc[0][col]) if not m.empty else None

# ─── FIGURE BUILDERS ──────────────────────────────────────────────────────────

def fig_salary(gross):
    if sal.empty:
        return go.Figure().update_layout(**LAYOUT, title="Salary data not loaded")
    row = tax_row(gross)
    ri, rf = row.eff_rate_italy_pct, row.eff_rate_finland_pct
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sal.gross_annual, y=sal.net_monthly_italy,
        name="Italy net/month", line=dict(color=C["pad"], width=2.5), mode="lines",
        hovertemplate="Gross: €%{x:,.0f}<br><b>Italy net: €%{y:,.0f}/mo</b><extra></extra>"))
    fig.add_trace(go.Scatter(x=sal.gross_annual, y=sal.net_monthly_finland,
        name="Finland net/month", line=dict(color=C["hel"], width=2.5), mode="lines",
        hovertemplate="Gross: €%{x:,.0f}<br><b>Finland net: €%{y:,.0f}/mo</b><extra></extra>"))
    fig.add_trace(go.Scatter(x=sal.gross_annual, y=sal.net_monthly_fi_ppp_adj,
        name="Finland PPP-adjusted/month",
        line=dict(color=C["hel"], width=1.5, dash="dash"), mode="lines",
        hovertemplate="Gross: €%{x:,.0f}<br><b>PPP-adj: €%{y:,.0f}/mo</b>"
                      "<br>(deflated by Eurostat PLI FI=122/IT=100)<extra></extra>"))
    fig.add_vline(x=gross, line_color=C["gold"], line_dash="dot", line_width=2,
        annotation_text=f"Your salary: €{gross:,.0f}",
        annotation_font_color=C["gold"], annotation_position="top right",
        annotation_bgcolor=C["surf"], annotation_bordercolor=C["gold"])
    fig.update_layout(**LO(height=360,
        title=dict(text=f"Net Monthly Income  ·  Italy eff. rate {ri:.1f}%  ·  Finland {rf:.1f}% (T1, T2)",
                   font=dict(size=13, color=C["text"])),
        xaxis_title="Gross annual salary (€)", yaxis_title="Net monthly income (€)",
        xaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"])))
    return fig

def fig_breakeven(gross, use_ppp=True):
    if be.empty:
        return go.Figure().update_layout(**LAYOUT, title="Break-even data not loaded")
    row  = nearest_row(be, "gross_italy", gross)
    bev  = row.fi_gross_ppp_parity if use_ppp else row.fi_gross_nominal_parity
    upl  = row.ppp_gross_uplift_pct if use_ppp else row.nominal_gross_uplift_pct
    bc   = "fi_gross_ppp_parity" if use_ppp else "fi_gross_nominal_parity"
    lbl  = "PPP" if use_ppp else "Nominal"
    fig  = go.Figure()
    fig.add_trace(go.Scatter(x=be.gross_italy, y=be.gross_italy, name="Equal salary",
        line=dict(color=C["border"], width=1, dash="dot"), mode="lines", hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=be.gross_italy, y=be[bc],
        name=f"Required Finnish gross ({lbl})",
        line=dict(color=C["hel"], width=2.5), mode="lines",
        fill="tonexty", fillcolor="rgba(30,111,173,0.07)",
        hovertemplate="Italian gross: €%{x:,.0f}<br>"
                      "<b>Required Finnish gross: €%{y:,.0f}</b><extra></extra>"))
    fig.add_vline(x=gross, line_color=C["gold"], line_dash="dot", line_width=2)
    fig.add_hline(y=bev,   line_color=C["hel"],  line_dash="dot", line_width=1)
    fig.add_annotation(x=gross, y=bev,
        text=f"  Required: €{bev:,.0f}<br>  (Italian salary +{upl:.1f}%)",
        showarrow=True, arrowhead=2, arrowcolor=C["gold"],
        font=dict(color=C["gold"], size=12),
        bgcolor=C["surf"], bordercolor=C["gold"], borderpad=6)
    fig.update_layout(**LO(height=360,
        title=dict(text=f"Break-Even Finnish Salary ({lbl} parity) (T3)",
                   font=dict(size=13, color=C["text"])),
        xaxis_title="Italian gross (€)", yaxis_title="Finnish gross required (€)",
        xaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"])))
    return fig

def fig_hicp():
    if hicp.empty: return go.Figure().update_layout(**LAYOUT, title="HICP not loaded")
    df = hicp.sort_values("diff_pct", ascending=True)
    cols = [C["warn"] if d>25 else C["hel"] if d>0 else C["good"] for d in df.diff_pct]
    fig = go.Figure(go.Bar(
        x=df.diff_pct, y=df.category, orientation="h",
        marker_color=cols,
        text=[f"+{d:.0f}%" if d>0 else f"{d:.0f}%" for d in df.diff_pct],
        textposition="outside", textfont=dict(size=11, color=C["text"]),
        customdata=list(zip(df.basket_weight_italy_pct, df.basket_weight_finland_pct)),
        hovertemplate="<b>%{y}</b><br>Finland vs Italy: %{text}<br>"
                      "Italy basket weight: %{customdata[0]:.1f}%<br>"
                      "Finland basket weight: %{customdata[1]:.1f}%<extra></extra>"))
    fig.add_vline(x=0, line_color=C["muted"], line_width=1)
    fig.update_layout(**LO(height=440, showlegend=False,
        title=dict(text="Price Level by Category — Finland vs Italy = 100 (T4)",
                   font=dict(size=13, color=C["text"])),
        xaxis_title="% above Italian price level",
        xaxis=dict(range=[-40,90], gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"])))
    return fig

def fig_numbeo(category, display_mode):
    if nmb.empty: return go.Figure().update_layout(**LAYOUT, title="Numbeo not loaded")
    df = nmb if category=="All" else nmb[nmb.category==category]
    df = df.sort_values("diff_pct", ascending=False)
    h  = max(320, len(df)*33+90)
    if display_mode == "pct":
        cols = [C["warn"] if d>0 else C["good"] for d in df.diff_pct]
        fig = go.Figure(go.Bar(
            y=df.item, x=df.diff_pct, orientation="h", marker_color=cols,
            text=[f"{d:+.0f}%" for d in df.diff_pct],
            textposition="outside", textfont=dict(size=11, color=C["text"]),
            cliponaxis=False,
            customdata=list(zip(df.price_eur_padova, df.price_eur_helsinki)),
            hovertemplate="<b>%{y}</b><br>Padova: €%{customdata[0]:.2f}<br>"
                          "Helsinki: €%{customdata[1]:.2f}<br>"
                          "<b>Difference: %{text}</b><extra></extra>"))
        fig.add_vline(x=0, line_color=C["muted"], line_width=1)
        fig.update_layout(**LO(height=h, showlegend=False,
            title=dict(text="Price Difference % — Helsinki vs Padova (T5)",
                       font=dict(size=13, color=C["text"])),
            xaxis_title="% difference (positive = Helsinki more expensive)",
            xaxis=dict(range=[df.diff_pct.min()-20, df.diff_pct.max()+25],
                       gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
            yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"],
                       autorange="reversed")))
    else:
        fig = go.Figure()
        fig.add_trace(go.Bar(y=df.item, x=df.price_eur_padova, name="Padova",
            orientation="h", marker_color=C["pad"],
            error_x=dict(type="data", symmetric=False,
                array=(df.price_high_padova-df.price_eur_padova).fillna(0),
                arrayminus=(df.price_eur_padova-df.price_low_padova).fillna(0),
                color=C["muted"], thickness=1.5, width=4),
            hovertemplate="<b>%{y}</b><br>Padova: €%{x:.2f}<extra></extra>"))
        fig.add_trace(go.Bar(y=df.item, x=df.price_eur_helsinki, name="Helsinki",
            orientation="h", marker_color=C["hel"],
            error_x=dict(type="data", symmetric=False,
                array=(df.price_high_helsinki-df.price_eur_helsinki).fillna(0),
                arrayminus=(df.price_eur_helsinki-df.price_low_helsinki).fillna(0),
                color=C["muted"], thickness=1.5, width=4),
            hovertemplate="<b>%{y}</b><br>Helsinki: €%{x:.2f}<br>"
                          "Diff: %{customdata:+.1f}%<extra></extra>",
            customdata=df.diff_pct))
        fig.update_layout(**LO(height=h, barmode="group",
            title=dict(text="Item Prices — Numbeo Scraped Data (T5)",
                       font=dict(size=13, color=C["text"])),
            xaxis_title="Price (€)",
            xaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
            yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"],
                       autorange="reversed"),
            legend=dict(orientation="h", y=1.02, x=0)))
    return fig

def fig_expenses(selected_ids, quantities):
    if not selected_ids:
        return go.Figure().update_layout(**LAYOUT, height=180,
            title=dict(text="Select activities above to see your personalised expense breakdown",
                       font=dict(size=13, color=C["muted"]))), 0, 0
    rows = []
    for aid, label, freq, dqty, nmatch in ACTIVITIES:
        if aid not in selected_ids: continue
        qty = quantities.get(aid, dqty)
        mult = FREQ_MULT.get(freq, 1.0)
        pp = nmb_price(nmatch, "price_eur_padova") or 0
        hp = nmb_price(nmatch, "price_eur_helsinki") or pp*1.3
        rows.append(dict(label=label, pad_mo=round(pp*qty*mult,2),
                         hel_mo=round(hp*qty*mult,2)))
    if not rows:
        return go.Figure().update_layout(**LAYOUT, height=180,
            title="No items"), 0, 0
    df = pd.DataFrame(rows)
    tot_p = round(df.pad_mo.sum(), 2)
    tot_h = round(df.hel_mo.sum(), 2)
    df["diff_pct"] = ((df.hel_mo-df.pad_mo)/df.pad_mo.replace(0,np.nan)*100).fillna(0)
    h = max(260, len(df)*36+80)
    fig = go.Figure()
    fig.add_trace(go.Bar(y=df.label, x=df.pad_mo, name="Padova",
        orientation="h", marker_color=C["pad"],
        hovertemplate="<b>%{y}</b><br>Padova: €%{x:.2f}/mo<extra></extra>"))
    fig.add_trace(go.Bar(y=df.label, x=df.hel_mo, name="Helsinki",
        orientation="h", marker_color=C["hel"],
        hovertemplate="<b>%{y}</b><br>Helsinki: €%{x:.2f}/mo<br>"
                      "Difference: %{customdata:+.1f}%<extra></extra>",
        customdata=df.diff_pct))
    fig.update_layout(**LO(height=h, barmode="group",
        title=dict(text=f"Monthly Expenses  ·  Padova €{tot_p:,.0f}  vs  Helsinki €{tot_h:,.0f}",
                   font=dict(size=13, color=C["text"])),
        xaxis_title="Monthly cost (€)",
        xaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"],
                   autorange="reversed"),
        legend=dict(orientation="h", y=1.02, x=0)))
    return fig, tot_p, tot_h

def fig_wages(occ_label, gross):
    if wages.empty: return go.Figure().update_layout(**LAYOUT, title="Wage data not loaded")
    fi_mo = (float(nearest_row(sal,"gross_annual",gross)["net_monthly_finland"])
             if not sal.empty else gross/12*0.72)
    df_w = wages.drop_duplicates("occ_label").sort_values("median", ascending=False)
    dot_cols = [C["gold"] if o==occ_label else C["hel"] for o in df_w["occ_label"]]
    dot_sz   = [14 if o==occ_label else 9  for o in df_w["occ_label"]]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df_w["median"], y=df_w["occ_label"], mode="markers",
        error_x=dict(type="data", symmetric=False,
            array=(df_w["p90"]-df_w["median"]).tolist(),
            arrayminus=(df_w["median"]-df_w["p10"]).tolist(),
            color=C["border"], thickness=6, width=0),
        marker=dict(color=dot_cols, size=dot_sz), name="Median (± P10–P90)",
        hovertemplate="<b>%{y}</b><br>Median: €%{x:,.0f}/mo<extra></extra>"))
    fig.add_trace(go.Scatter(x=df_w["median"], y=df_w["occ_label"], mode="markers",
        error_x=dict(type="data", symmetric=False,
            array=(df_w["p75"]-df_w["median"]).tolist(),
            arrayminus=(df_w["median"]-df_w["p25"]).tolist(),
            color="rgba(30,111,173,0.35)", thickness=10, width=0),
        marker=dict(opacity=0, size=1), name="IQR P25–P75", hoverinfo="skip"))
    sel = df_w[df_w["occ_label"]==occ_label]
    band = "—"
    if not sel.empty:
        s = sel.iloc[0]
        if   fi_mo<s["p10"]:    band="Below P10 (bottom 10%)"
        elif fi_mo<s["p25"]:    band="P10–P25 (lower quartile)"
        elif fi_mo<s["median"]: band="P25–Median"
        elif fi_mo<s["p75"]:    band="Median–P75"
        elif fi_mo<s["p90"]:    band="P75–P90 (top 25%)"
        else:                   band="Above P90 (top 10%)"
    fig.add_vline(x=fi_mo, line_color=C["gold"], line_dash="dot", line_width=2,
        annotation_text=f"Your FI net: €{fi_mo:,.0f}/mo<br><b>{band}</b>",
        annotation_font_color=C["gold"], annotation_position="top right",
        annotation_bgcolor=C["surf"], annotation_bordercolor=C["gold"])
    fig.update_layout(**LO(height=370,
        title=dict(text=f"Finnish Wage Distribution — {occ_label}  ·  {band} (T6)",
                   font=dict(size=12, color=C["text"])),
        xaxis_title="Monthly net income (€/month)",
        xaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"])))
    return fig

def fig_urban():
    if ua is None or ua.empty:
        return go.Figure().update_layout(**LAYOUT, title="Urban Audit data not loaded")
    df = ua.copy().sort_values("diff_pct", ascending=True)
    X_CAP = 100.0
    df["diff_clipped"] = df["diff_pct"].clip(-X_CAP, X_CAP)
    df["is_clipped"]   = df["diff_pct"].abs() > X_CAP
    df["bar_text"] = df.apply(
        lambda r: f"{r.diff_pct:+.0f}%▶" if r.is_clipped else f"{r.diff_pct:+.0f}%", axis=1)
    bar_cols = [C["hel"] if f=="Helsinki" else C["pad"] for f in df["favours"]]
    clipped  = df[df["is_clipped"]]["indicator"].tolist()
    fig = go.Figure(go.Bar(
        x=df["diff_clipped"], y=df["indicator"], orientation="h",
        marker_color=bar_cols,
        text=df["bar_text"], textposition="inside",
        textfont=dict(size=11, color="white"), cliponaxis=False,
        customdata=list(zip(df.padova.round(2), df.helsinki.round(2),
                            df.unit, df.favours, df.diff_pct.round(1))),
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Padova: %{customdata[0]} %{customdata[2]}<br>"
            "Helsinki: %{customdata[1]} %{customdata[2]}<br>"
            "Actual difference: %{customdata[4]:+.1f}%<br>"
            "Better city: <b>%{customdata[3]}</b><extra></extra>")))
    fig.add_vline(x=0, line_color=C["muted"], line_width=1.5)
    legend_note = (
        f"<span style='color:{C['pad']}'>■</span> Padova performs better  "
        f"<span style='color:{C['hel']}'>■</span> Helsinki performs better<br>"
        f"Bars capped at ±100%; actual value shown inside bar (▶ = truncated)"
    )
    if clipped:
        legend_note += f"<br>Truncated indicators: {', '.join(clipped)}"
    fig.add_annotation(xref="paper", yref="paper", x=0, y=-0.12,
        text=legend_note, showarrow=False, align="left",
        font=dict(size=9, color=C["muted"]))
    fig.update_layout(**LO(height=530, showlegend=False,
        title=dict(text="Urban Audit City Indicators — Eurostat 2021–2023 (T7)",
                   font=dict(size=13, color=C["text"])),
        xaxis_title="% difference (Helsinki vs Padova, bars capped at ±100%)",
        margin=dict(l=220, r=100, t=48, b=100),
        xaxis=dict(range=[-80,120], gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"])))
    return fig

def fig_estat(choice):
    mapping = {
        "income": (e_inc, "Mean equivalised income", "€"),
        "employ": (e_emp, "Employment rate 20–64", "%"),
        "hous_ob": (e_hou, "Housing overburden rate", "%"),
        "hous_cs": (e_hou2, "Housing cost / income", "%")
    }
    df, label, unit = mapping.get(choice, (pd.DataFrame(), "—", ""))
    if df is None or df.empty:
        return go.Figure().update_layout(**LO(height=200, title=f"{label} not loaded"))

    # Identify columns
    gc = next((c for c in df.columns if c.lower() in ("geo", "ref_area", "cities")), None)
    vc = next((c for c in df.columns if c.lower() in ("obs_value", "value", "values")), None)
    tc = next((c for c in df.columns if c.lower() in ("time", "year", "time_period")), None)
    if not gc or not vc:
        return go.Figure().update_layout(title="Column structure unrecognised")

    sub = df.copy()
    sub[vc] = pd.to_numeric(sub[vc], errors="coerce")
    sub = sub.dropna(subset=[vc])

    # Keep only Italian and Finnish regions
    sub = sub[sub[gc].str.startswith(("IT", "FI"), na=False)]

    # Ensure time is numeric
    if tc:
        sub[tc] = sub[tc].astype(int)

    # Create a scatter plot over time, colored by region
    fig = go.Figure()
    regions = sub[gc].unique()
    for region in regions:
        reg_data = sub[sub[gc] == region]
        if not reg_data.empty:
            fig.add_trace(go.Scatter(
                x=reg_data[tc], y=reg_data[vc],
                mode='lines+markers',
                name=region,
                hovertemplate=f"<b>{region}</b><br>Year: %{{x}}<br>{label}: %{{y:.1f}} {unit}<extra></extra>"
            ))
    fig.update_layout(**LO(height=450, showlegend=True,
        title=dict(text=f"{label} – time series (IT & FI regions)", font=dict(size=13, color=C["text"])),
        xaxis_title="Year", yaxis_title=unit,
        xaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"])))
    return fig

# ─── DESIGN STAGE: WHAT IF? ───────────────────────────────────────────────────
def fig_whatif(gross_it, gross_fi_offer):
    """
    Side-by-side comparison of a specific Italian salary vs a specific Finnish offer.
    Implements the Design stage: generating and exploring a concrete alternative.
    """
    if sal.empty:
        return go.Figure().update_layout(**LAYOUT, title="Salary data not loaded")
    row_it = nearest_row(sal, "gross_annual", gross_it)
    row_fi = nearest_row(sal, "gross_annual", gross_fi_offer)
    row_be = nearest_row(be,  "gross_italy",   gross_it) if not be.empty else None

    ni_mo  = row_it.net_monthly_italy
    nf_mo  = row_fi.net_monthly_finland
    pp_mo  = row_fi.net_monthly_fi_ppp_adj
    bev    = row_be.fi_gross_ppp_parity if row_be is not None else gross_it*1.28
    uplift = ((gross_fi_offer/bev)-1)*100

    categories = ["Italy net/month", "Finland net/month", "Finland net PPP-adj"]
    italian    = [ni_mo, None, None]
    finnish    = [None, nf_mo, pp_mo]

    fig = go.Figure()
    fig.add_trace(go.Bar(x=categories, y=[ni_mo, ni_mo, ni_mo],
        name="Italy (your current salary)", marker_color=C["pad"],
        hovertemplate="<b>%{x}</b><br>Italy benchmark: €%{y:,.0f}/mo<extra></extra>"))
    fig.add_trace(go.Bar(x=categories, y=[nf_mo, nf_mo, pp_mo],
        name=f"Finland (€{gross_fi_offer:,.0f} offer)", marker_color=C["hel"],
        hovertemplate="<b>%{x}</b><br>Finnish offer: €%{y:,.0f}/mo<extra></extra>"))


    # Verdict line
    verdict_col = C["good"] if pp_mo > ni_mo else C["warn"]
    verdict_txt = (
        f"<b style='color:{verdict_col}'>{'✓ Finnish offer exceeds your current purchasing power' if pp_mo > ni_mo else '✗ Finnish offer does not match purchasing power'}</b><br>"
        f"PPP-adjusted Finnish net: €{pp_mo:,.0f}/mo vs Italy: €{ni_mo:,.0f}/mo<br>"
        f"You are {'above' if gross_fi_offer >= bev else 'below'} break-even (€{bev:,.0f} required)<br>"
        f"Offer is {abs(uplift):.1f}% {'above' if uplift>=0 else 'below'} PPP break-even"
    )


    fig.update_layout(**LO(height=380, barmode="group",
        title=dict(text=f"What If? — IT €{gross_it:,.0f} vs FI Offer €{gross_fi_offer:,.0f}",
                   font=dict(size=13, color=C["text"])),
        xaxis_title="Comparison metric", yaxis_title="Monthly net income (€)",
        yaxis=dict(tickprefix="€", gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        xaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        legend=dict(orientation="h", y=1.02, x=0),
        annotations=[dict(xref="paper", yref="paper", x=0, y=-0.22,
            text=verdict_txt, showarrow=False, align="left",
            font=dict(size=11, color=C["text"]))]))
    return fig

# ─── CHOICE STAGE: WEIGHTED VERDICT ──────────────────────────────────────────
def fig_choice(weights, gross_it, gross_fi_offer):
    """
    Implements the Choice stage: user-weighted criteria evaluation.
    weights: dict {crit_id: 0–5}
    Returns: fig, verdict_text
    """
    # Dynamic salary criterion: derived from actual tax computation
    if not sal.empty:
        row_it = nearest_row(sal, "gross_annual", gross_it)
        row_fi = nearest_row(sal, "gross_annual", gross_fi_offer)
        ni_mo  = row_it.net_monthly_italy
        nf_mo  = row_fi.net_monthly_finland
        pp_mo  = row_fi.net_monthly_fi_ppp_adj
        max_net = max(ni_mo, nf_mo, 1)
        pad_sal = round(ni_mo / max_net * 10, 1)
        hel_sal = round(pp_mo / max_net * 10, 1)  # PPP-adjusted
    else:
        pad_sal, hel_sal = 5.0, 5.0
    criteria_scores = compute_criteria_scores(ua)
    if criteria_scores is None:
        return go.Figure().update_layout(title="Urban Audit data unavailable"), "No data", C["warn"]

    # Build the criteria list for the chart
    chart_criteria = [("salary", "💰 Salary (PPP-adjusted)", pad_sal, hel_sal, "Tax model")]
    for cid, scores in criteria_scores.items():
        if cid == "col":
            lbl = "🛒 Cost of living"
        elif cid == "qol":
            lbl = "😊 Quality of life"
        # ... map other labels ...
        chart_criteria.append((cid, f"{CRITERION_LABELS.get(cid, cid)}", scores["padova"], scores["helsinki"], "Urban Audit"))

    # The rest of the function proceeds with chart_criteria
    all_criteria = [("salary", "💰 Salary (PPP-adjusted)", pad_sal, hel_sal,
                     "Tax-adjusted net income, deflated by Eurostat PLI")] + \
                   [(cid, lbl, ps, hs, src) for cid,lbl,ps,hs,src in
                    [(c[0],c[1],c[2],c[3],c[4]) for c in CRITERIA]]

    labels, pad_scores, hel_scores, ws = [], [], [], []
    for cid, lbl, ps, hs, _ in chart_criteria: #instead of all_criteria
        w = weights.get(cid, 3)
        labels.append(lbl);  pad_scores.append(ps)
        hel_scores.append(hs); ws.append(w)

    ws_arr = np.array(ws, dtype=float)
    pad_arr= np.array(pad_scores, dtype=float)
    hel_arr= np.array(hel_scores, dtype=float)

    if ws_arr.sum() == 0:
        return go.Figure().update_layout(**LAYOUT, height=200,
            title="Set at least one criterion weight above zero"), "Set weights to see verdict"

    pad_weighted = float((ws_arr * pad_arr).sum() / ws_arr.sum())
    hel_weighted = float((ws_arr * hel_arr).sum() / ws_arr.sum())
    diff = hel_weighted - pad_weighted

    # Build comparison bar chart
    fig = go.Figure()
    # Unweighted individual scores
    fig.add_trace(go.Scatter(
        x=pad_arr, y=labels, mode="markers",
        marker=dict(color=C["pad"], size=10, symbol="circle"),
        name="Padova score", xaxis="x",
        hovertemplate="<b>%{y}</b><br>Padova: %{x:.1f}/10<extra></extra>"))
    fig.add_trace(go.Scatter(
        x=hel_arr, y=labels, mode="markers",
        marker=dict(color=C["hel"], size=10, symbol="diamond"),
        name="Helsinki score", xaxis="x",
        hovertemplate="<b>%{y}</b><br>Helsinki: %{x:.1f}/10<extra></extra>"))
    # Connect pairs with lines
    for i in range(len(labels)):
        fig.add_shape(type="line",
            x0=pad_arr[i], x1=hel_arr[i], y0=i, y1=i,
            line=dict(color=C["hel"] if hel_arr[i]>pad_arr[i] else C["pad"],
                      width=max(1, ws[i]*1.5), dash="solid" if ws[i]>0 else "dot"),
            opacity=0.6)
    # Weighted totals
    verdict_col = C["good"] if diff > 0 else C["warn"] if diff < -0.3 else C["gold"]
    win_city = "Helsinki" if diff > 0 else "Padova" if diff < 0 else "Tied"
    verdict = (
        f"Based on your priorities: <b>{win_city}</b> scores "
        f"{hel_weighted:.1f}/10 vs {pad_weighted:.1f}/10  "
        f"({'Helsinki leads by' if diff>0 else 'Padova leads by'} {abs(diff):.1f} points)"
    )
    fig.add_annotation(xref="paper", yref="paper", x=0, y=-0.14,
        text=f"Weighted totals → Padova: <b>{pad_weighted:.1f}/10</b>  "
             f"Helsinki: <b>{hel_weighted:.1f}/10</b>  |  "
             f"{'Line width = importance weight (thicker = more important). '}"
             f"Dashed = weight=0 (ignored).",
        showarrow=False, align="left", font=dict(size=9, color=C["muted"]))
    fig.update_layout(**LO(height=max(380, len(labels)*38+100),
        title=dict(text="Weighted Criteria Comparison — Your Priorities Applied (T9)",
                   font=dict(size=13, color=C["text"])),
        xaxis_title="Score (0 = worst, 10 = best)",
        xaxis=dict(range=[0,11], gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        yaxis=dict(gridcolor=C["border"], zeroline=False, linecolor=C["border"]),
        margin=dict(l=240, r=40, t=48, b=100),
        legend=dict(orientation="h", y=1.02, x=0)))
    return fig, verdict, verdict_col

# ─── DASH APP ─────────────────────────────────────────────────────────────────
app = dash.Dash(__name__, title="Padova ↔ Helsinki Decision Tool")
app.config.suppress_callback_exceptions = True

S = dict(
    outer ={"backgroundColor":C["bg"],"minHeight":"100vh","color":C["text"],
            "fontFamily":"IBM Plex Sans,sans-serif"},
    header={"background":C["surf"],"borderBottom":f"1px solid {C['border']}",
            "padding":"16px 28px","display":"flex","justifyContent":"space-between",
            "alignItems":"center","flexWrap":"wrap","gap":"12px",
            "boxShadow":"0 1px 4px rgba(0,0,0,.08)"},
    h1    ={"fontFamily":"Playfair Display,serif","fontSize":"24px","color":C["text"],
            "margin":"0","letterSpacing":".01em"},
    sub   ={"color":C["muted"],"fontSize":"10px","letterSpacing":".1em",
            "textTransform":"uppercase","marginTop":"3px"},
    badge_pad={"background":"rgba(184,92,40,.1)","color":C["pad"],
               "border":f"1px solid rgba(184,92,40,.3)","borderRadius":"4px",
               "padding":"3px 12px","fontSize":"11px","fontWeight":"700",
               "letterSpacing":".08em","textTransform":"uppercase"},
    badge_hel={"background":"rgba(30,111,173,.1)","color":C["hel"],
               "border":f"1px solid rgba(30,111,173,.3)","borderRadius":"4px",
               "padding":"3px 12px","fontSize":"11px","fontWeight":"700",
               "letterSpacing":".08em","textTransform":"uppercase"},
    ctrl  ={"background":C["panel"],"borderBottom":f"1px solid {C['border']}",
            "padding":"14px 28px","display":"flex","gap":"24px",
            "flexWrap":"wrap","alignItems":"center"},
    ctrl_g={"display":"flex","flexDirection":"column","gap":"5px","flex":"1","minWidth":"180px"},
    lbl   ={"color":C["muted"],"fontSize":"10px","letterSpacing":".1em",
            "textTransform":"uppercase","fontWeight":"600"},
    sal_v ={"color":C["gold"],"fontSize":"22px","fontWeight":"800"},
    kpi_s ={"display":"grid","gridTemplateColumns":"repeat(5,1fr)",
            "gap":"1px","background":C["border"]},
    kpi   ={"background":C["surf"],"padding":"12px 16px"},
    kpi_l ={"color":C["muted"],"fontSize":"9px","letterSpacing":".1em",
            "textTransform":"uppercase","marginBottom":"5px","fontWeight":"600"},
    kpi_v ={"fontSize":"17px","fontWeight":"800"},
    tab_s ={"background":C["surf"],"borderBottom":f"2px solid {C['border']}",
            "padding":"0 28px","display":"flex","gap":"0px",
            "overflowX":"auto"},
    note  ={"color":C["muted"],"fontSize":"12px","lineHeight":"1.7",
            "padding":"10px 16px","background":C["panel"],
            "border":f"1px solid {C['border']}","borderRadius":"6px",
            "marginBottom":"14px"},
    footer={"padding":"14px 28px","color":C["muted"],"fontSize":"10px",
            "lineHeight":"1.8","borderTop":f"1px solid {C['border']}",
            "background":C["surf"]},
    input ={"background":C["surf"],"border":f"1px solid {C['border']}",
            "color":C["text"],"borderRadius":"5px","padding":"6px 10px",
            "fontFamily":"inherit","fontSize":"13px","width":"100%"},
    dd    ={"background":C["surf"],"color":"#000","width":"260px"},
    wt_sl ={"accentColor":C["hel"]},
)

TABS_DEF = [
    ("salary",   "💼 Salary"),
    ("expenses", "🧾 My Expenses"),
    ("prices",   "🛒 Prices"),
    ("wages",    "📊 Wages"),
    ("city",     "🏙 City Quality"),
    ("whatif",   "🔀 What If?"),
    ("decision", "⚖️ My Decision"),
    ("datasets", "📁 Datasets"),
    ("profile",  "📈 My Profile"),
]

def tbtn(tid, lbl, active=False):
    return html.Button(lbl, id=f"tab-{tid}", n_clicks=1 if active else 0,
        style={"border":"none","borderBottom":f"3px solid {C['gold']}" if active
               else "3px solid transparent","background":"none",
               "cursor":"pointer","padding":"11px 14px",
               "color":C["gold"] if active else C["muted"],
               "fontFamily":"inherit","fontSize":"12px","letterSpacing":".04em",
               "fontWeight":"700" if active else "400","whiteSpace":"nowrap"})

def kpi_card(lbl, kid, col=None):
    return html.Div([html.Div(lbl, style=S["kpi_l"]),
                     html.Div("—", id=kid, style={**S["kpi_v"],"color":col or C["text"]})],
                    style=S["kpi"])

def notediv(text):
    return html.Div(text, style=S["note"])

def pane(pid, children):
    return html.Div(children, id=f"pane-{pid}",
                    style={"display":"block" if pid=="salary" else "none",
                           "padding":"20px 28px"})

def wt_slider(crit_id, label, source):
    return html.Div([
        html.Div([
            html.Span(label, style={"fontSize":"13px","color":C["text"],"flex":"1"}),
            html.Span(id=f"wt-val-{crit_id}", children="3",
                      style={"color":C["hel"],"fontWeight":"700","fontSize":"13px",
                             "minWidth":"20px","textAlign":"right"}),
        ], style={"display":"flex","justifyContent":"space-between","marginBottom":"3px"}),
        dcc.Slider(id=f"wt-{crit_id}", min=0, max=5, step=1, value=3,
            marks={0:"0 (ignore)", 5:"5 (crucial)"},
            tooltip={"placement":"bottom","always_visible":False}),
        html.Div(source, style={"color":C["muted"],"fontSize":"10px","marginTop":"2px"}),
    ], style={"marginBottom":"18px"})

app.layout = html.Div([

    html.Div([
        html.Div([html.H1("Padova ↔ Helsinki Decision Tool", style=S["h1"]),
                  html.Div("Relocation decision support · 2023–2024 data", style=S["sub"])]),
        html.Div([html.Span("Padova IT",   style=S["badge_pad"]),
                  html.Span("Helsinki FI", style=S["badge_hel"])],
                 style={"display":"flex","gap":"8px"}),
    ], style=S["header"]),

    # CONTROLS
    html.Div([
        html.Div([
            html.Div("Your gross annual salary (Italy)", style=S["lbl"]),
            html.Div("€35,000", id="sal-display", style=S["sal_v"]),
            dcc.Slider(id="sal-slider", min=15000, max=120000, step=500, value=35000,
                marks={15000:"€15k",35000:"€35k",60000:"€60k",90000:"€90k",120000:"€120k"},
                tooltip={"placement":"bottom","always_visible":False}),
        ], style={**S["ctrl_g"],"flex":"3"}),
        html.Div([
            html.Div("Break-even mode", style=S["lbl"]),
            dcc.RadioItems(id="be-mode",
                options=[{"label":" PPP parity (real)","value":"ppp"},
                         {"label":" Nominal","value":"nominal"}],
                value="ppp", inline=True,
                labelStyle={"marginRight":"14px","color":C["text"],"fontSize":"13px"},
                inputStyle={"accentColor":C["gold"],"marginRight":"5px"}),
        ], style=S["ctrl_g"]),
    ], style=S["ctrl"]),

    # KPI STRIP
    html.Div([
        kpi_card("Italy net / month",        "kpi-it",  C["pad"]),
        kpi_card("Finland net / month",      "kpi-fi",  C["hel"]),
        kpi_card("Finland PPP-adj / month",  "kpi-pp",  C["hel"]),
        kpi_card("Break-even Finnish gross", "kpi-be",  C["gold"]),
        kpi_card("Monthly disposable diff",  "kpi-di"),
    ], style=S["kpi_s"]),

    html.Div([tbtn(tid, lbl, active=(tid=="salary")) for tid,lbl in TABS_DEF],
             style=S["tab_s"]),

    dcc.Store(id="active-tab", data="salary"),
    dcc.Store(id="expense-totals", data={"pad":0,"hel":0}),

    # ── SALARY ──
    pane("salary", [
        notediv("T1 Compare · T2 Identify: net income curves as salary changes. "
                "Dashed blue = Finnish net adjusted for higher Finnish prices (PLI). "
                "T3 Locate: the Finnish salary needed to match your Italian purchasing power."),
        html.Div([fig_salary.__doc__,
            dcc.Graph(id="fig-salary"),
            dcc.Graph(id="fig-breakeven"),
        ], style={"display":"grid","gridTemplateColumns":"1fr 1fr","gap":"16px"}),
    ]),

    # ── EXPENSES ──
    pane("expenses", [
        notediv("T5 Personalised expenses: select what you spend money on in Italy — "
                "Helsinki prices auto-filled from Numbeo scraped data (April 2026). "
                "Monthly totals feed into the disposable income KPI above."),
        html.Div([
            html.Div([
                html.Div("Select your regular activities:", style={**S["lbl"],"marginBottom":"10px"}),
                dcc.Checklist(id="act-checklist",
                    options=[{"label":f"  {lbl} ({freq})", "value":aid}
                             for aid,lbl,freq,_,_ in ACTIVITIES],
                    value=["coffee","lunch_r","transport","rent_1br"],
                    labelStyle={"display":"block","marginBottom":"10px",
                                "color":C["text"],"fontSize":"13px"},
                    inputStyle={"accentColor":C["hel"],"marginRight":"8px",
                                "width":"16px","height":"16px"}),
                html.Hr(style={"borderColor":C["border"],"margin":"14px 0"}),
                html.Div("Quantities:", style={**S["lbl"],"marginBottom":"10px"}),
                html.Div(id="qty-controls"),
            ], style={"flex":"1","minWidth":"240px"}),
            html.Div([
                dcc.Graph(id="fig-expenses"),
                html.Div(id="expense-cards"),
            ], style={"flex":"2","minWidth":"380px"}),
        ], style={"display":"flex","gap":"24px","flexWrap":"wrap"}),
    ]),

    # ── PRICES ──
    pane("prices", [
        notediv("T4 HICP: structural price levels by category (Italy=100 baseline). "
                "T5 Numbeo items: toggle between absolute € prices and % difference — "
                "the % view puts beer and rent on the same scale."),
        html.Div([
            dcc.Graph(id="fig-hicp"),
            html.Div([
                html.Div([
                    html.Span("Category:", style={**S["lbl"],"marginRight":"8px"}),
                    dcc.Dropdown(id="nmb-cat",
                        options=[{"label":c,"value":c} for c in NUMBEO_CATS],
                        value="All", clearable=False, style=S["dd"]),
                    html.Span("  Display:", style={**S["lbl"],"marginLeft":"14px","marginRight":"8px"}),
                    dcc.RadioItems(id="nmb-mode",
                        options=[{"label":" € prices","value":"absolute"},
                                 {"label":" % difference","value":"pct"}],
                        value="absolute", inline=True,
                        labelStyle={"color":C["text"],"fontSize":"12px","marginRight":"10px"},
                        inputStyle={"accentColor":C["hel"],"marginRight":"4px"}),
                ], style={"display":"flex","alignItems":"center","marginBottom":"12px",
                          "flexWrap":"wrap","gap":"6px"}),
                dcc.Graph(id="fig-numbeo"),
            ]),
        ], style={"display":"grid","gridTemplateColumns":"1fr 1fr","gap":"16px"}),
    ]),

    # ── WAGES ──
    pane("wages", [
        notediv("T6 Finnish wage distribution by occupation. "
                "Gold marker and line = your Finnish net income at the selected salary. "
                "Thick bar = middle 50% (P25–P75). Thin bar = full P10–P90 range."),
        html.Div([
            html.Span("Occupation group:", style={**S["lbl"],"marginRight":"8px"}),
            dcc.Dropdown(id="occ-sel",
                options=[{"label":o,"value":o} for o in OCC_LABELS],
                value=OCC_LABELS[0] if OCC_LABELS else "All occupations",
                clearable=False, style={**S["dd"],"width":"300px","marginBottom":"12px"}),
        ], style={"display":"flex","alignItems":"center","gap":"8px"}),
        dcc.Graph(id="fig-wages"),
    ]),

    # ── CITY QUALITY ──
    pane("city", [
        notediv("T7 Urban Audit indicators: diverging bars centred at zero. "
                "Percentage inside the bar = actual difference (▶ = bar truncated at ±100%). "
                "Colour encodes which city performs better for each indicator."),
        dcc.Graph(id="fig-urban"),
    ]),

    # ── WHAT IF? (Design stage) ──
    pane("whatif", [
        notediv("Design stage (Oral et al., 2023): compare a specific Finnish job offer "
                "against your current Italian salary. The tool computes whether the offer "
                "exceeds the purchasing-power break-even threshold."),
        html.Div([
            html.Div([
                html.Div("Your Italian gross salary:", style={**S["lbl"],"marginBottom":"6px"}),
                html.Div("↑ Set with slider at top", style={"color":C["muted"],"fontSize":"12px"}),
                html.Br(),
                html.Div("Finnish job offer (gross annual €):", style={**S["lbl"],"marginBottom":"8px"}),
                dcc.Slider(id="fi-offer", min=20000, max=150000, step=1000, value=50000,
                    marks={20000:"€20k",50000:"€50k",80000:"€80k",
                           110000:"€110k",150000:"€150k"},
                    tooltip={"placement":"bottom","always_visible":True}),
                html.Div(id="whatif-offer-display",
                         style={"color":C["hel"],"fontSize":"20px","fontWeight":"700",
                                "marginTop":"10px"}),
            ], style={"flex":"1","minWidth":"260px"}),
            html.Div([dcc.Graph(id="fig-whatif")], style={"flex":"2","minWidth":"380px"}),
        ], style={"display":"flex","gap":"24px","flexWrap":"wrap","alignItems":"flex-start"}),
    ]),

    # ── MY DECISION (Choice stage) ──
    pane("decision", [
        notediv("Choice stage (Oral et al., 2023): assign importance weights to what matters "
                "to you. The tool computes a weighted score for each city and delivers a "
                "personalised verdict. Line thickness = your weight for that criterion."),
        html.Div([
            # Weights panel
            html.Div([
                html.Div("How important is each factor to you?",
                         style={**S["lbl"],"marginBottom":"16px","fontSize":"12px"}),
                wt_slider("salary","💰 Salary & purchasing power",
                          "Tax-adjusted net income, deflated by Eurostat PLI"),
                *[wt_slider(cid, lbl, src)
                  for cid,lbl,_,_,src in CRITERIA],
                html.Hr(style={"borderColor":C["border"],"margin":"8px 0"}),
                html.Div("Finnish offer used for salary score:", style={**S["lbl"],"marginBottom":"6px"}),
                html.Div("↑ Set in the What If? tab", style={"color":C["muted"],"fontSize":"12px"}),
            ], style={"flex":"1","minWidth":"260px","background":C["panel"],
                      "borderRadius":"8px","padding":"20px",
                      "border":f"1px solid {C['border']}"}),
            # Chart + verdict
            html.Div([
                dcc.Graph(id="fig-choice"),
                html.Div(id="verdict-box",
                         style={"marginTop":"16px","padding":"16px 20px",
                                "borderRadius":"8px","border":f"1px solid {C['border']}",
                                "background":C["surf"],"fontSize":"14px","lineHeight":"1.7"}),
            ], style={"flex":"2","minWidth":"400px"}),
        ], style={"display":"flex","gap":"24px","flexWrap":"wrap","alignItems":"flex-start"}),
    ]),

    # ── DATASETS ──
    pane("datasets", [
        notediv("T8 Eurostat regional data (years ≥ 2014, IT/FI NUTS regions only). "
                "Blue bars = Finland, terracotta = Italy."),
        html.Div([
            html.Span("Dataset:", style={**S["lbl"],"marginRight":"8px"}),
            dcc.Dropdown(id="estat-choice",
                options=[{"label":"Mean equivalised income","value":"income"},
                         {"label":"Employment rate 20–64","value":"employ"},
                         {"label":"Housing overburden rate","value":"hous_ob"},
                         {"label":"Housing cost / income","value":"hous_cs"}],
                value="income", clearable=False, style={**S["dd"],"width":"280px"}),
        ], style={"display":"flex","alignItems":"center","gap":"8px","marginBottom":"14px"}),
        dcc.Graph(id="fig-estat"),
    ]),

    *log_stores(),
    log_panel(),

    html.Div([
    "Data from Eurostat, Statistics Finland, Numbeo, and OECD. ",
    "Framework: Oral et al. (2023) IEEE TVCG.",
    ], style=S["footer"]),

], style=S["outer"])

# ─── CALLBACKS ────────────────────────────────────────────────────────────────

register_callbacks(app)

@app.callback(Output("sal-display","children"), Input("sal-slider","value"))
def _sal_disp(g): return f"€{int(g):,}"

@app.callback(Output("whatif-offer-display","children"), Input("fi-offer","value"))
def _fi_disp(v): return f"Finnish offer: €{int(v):,}"

# TAB SWITCHING — pane visibility + button styles
@app.callback(
    *[Output(f"pane-{tid}","style")  for tid,_ in TABS_DEF],
    *[Output(f"tab-{tid}","style")   for tid,_ in TABS_DEF],
    *[Input(f"tab-{tid}","n_clicks") for tid,_ in TABS_DEF])
def switch_tabs(*clicks):
    ctx = callback_context
    active = "salary"
    if ctx.triggered:
        active = ctx.triggered[0]["prop_id"].split("-",1)[1].split(".")[0]
    vis = {"display":"block","padding":"20px 28px"}
    hid = {"display":"none"}
    def bstyle(tid):
        on = (tid==active)
        return {"border":"none","borderBottom":f"3px solid {C['gold']}" if on
                else "3px solid transparent","background":"none",
                "cursor":"pointer","padding":"11px 14px",
                "color":C["gold"] if on else C["muted"],
                "fontFamily":"inherit","fontSize":"12px","letterSpacing":".04em",
                "fontWeight":"700" if on else "400","whiteSpace":"nowrap"}
    return *[vis if tid==active else hid for tid,_ in TABS_DEF], \
           *[bstyle(tid) for tid,_ in TABS_DEF]

# KPI STRIP
@app.callback(
    Output("kpi-it","children"), Output("kpi-fi","children"),
    Output("kpi-pp","children"), Output("kpi-be","children"),
    Output("kpi-di","children"), Output("kpi-di","style"),
    Input("sal-slider","value"), Input("expense-totals","data"))
def _kpis(gross, totals):
    ep = (totals or {}).get("pad",0); eh = (totals or {}).get("hel",0)
    k = get_kpis(gross, ep, eh)
    di_raw = k[4]
    di_str, di_num = di_raw if isinstance(di_raw,tuple) else (di_raw, 0)
    return k[0],k[1],k[2],k[3],di_str, \
           {**S["kpi_v"],"color":C["good"] if di_num>=0 else C["warn"]}

# SALARY + BREAKEVEN
@app.callback(Output("fig-salary","figure"),   Input("sal-slider","value"))
def _s(g): return fig_salary(g)

@app.callback(Output("fig-breakeven","figure"),
              Input("sal-slider","value"), Input("be-mode","value"))
def _be(g,m): return fig_breakeven(g, use_ppp=(m=="ppp"))

# EXPENSES
@app.callback(Output("qty-controls","children"), Input("act-checklist","value"))
def _qty_ctrl(sel):
    if not sel: return html.Div("Tick an activity above",
                                 style={"color":C["muted"],"fontSize":"12px"})
    return [html.Div([
        html.Span(lbl, style={"color":C["text"],"fontSize":"12px","flex":"1"}),
        dcc.Input(id={"type":"qty","index":aid}, type="number",
            value=dqty, min=0.25, step=0.25,
            style={**S["input"],"width":"60px","textAlign":"right","padding":"3px 6px"}),
        html.Span(f"× {freq.lower()}",
                  style={"color":C["muted"],"fontSize":"11px","minWidth":"60px"}),
    ], style={"display":"flex","alignItems":"center","gap":"8px","marginBottom":"8px"})
    for aid,lbl,freq,dqty,_ in ACTIVITIES if aid in sel]

@app.callback(
    Output("fig-expenses","figure"),
    Output("expense-cards","children"),
    Output("expense-totals","data"),
    Input("act-checklist","value"),
    Input({"type":"qty","index":ALL},"value"),
    Input({"type":"qty","index":ALL},"id"))
def _expenses(sel, qvals, qids):
    qty = {qid["index"]:float(qv) for qid,qv in zip(qids,qvals) if qv is not None}
    f,tp,th = fig_expenses(sel or [], qty)
    diff = th-tp; pct = round((diff/tp*100) if tp else 0, 1)
    cards = html.Div([
        html.Div([html.Div("Padova total/month",style=S["kpi_l"]),
                  html.Div(f"€{tp:,.0f}",style={**S["kpi_v"],"color":C["pad"]})],
                 style={**S["kpi"],"flex":"1","border":f"2px solid {C['pad']}44",
                        "borderRadius":"8px","marginTop":"12px"}),
        html.Div([html.Div("Helsinki total/month",style=S["kpi_l"]),
                  html.Div(f"€{th:,.0f}",style={**S["kpi_v"],"color":C["hel"]})],
                 style={**S["kpi"],"flex":"1","border":f"2px solid {C['hel']}44",
                        "borderRadius":"8px","marginTop":"12px"}),
        html.Div([html.Div("Expense difference",style=S["kpi_l"]),
                  html.Div(f"+{pct:.0f}% higher in Helsinki" if diff>0
                           else f"{abs(pct):.0f}% lower in Helsinki",
                           style={**S["kpi_v"],"color":C["warn"] if diff>0 else C["good"],
                                  "fontSize":"13px"})],
                 style={**S["kpi"],"flex":"1","border":f"1px solid {C['border']}",
                        "borderRadius":"8px","marginTop":"12px"}),
    ], style={"display":"flex","gap":"10px","flexWrap":"wrap"})
    return f, cards, {"pad":tp,"hel":th}

# PRICES
@app.callback(Output("fig-hicp","figure"), Input("sal-slider","value"))
def _hicp(_): return fig_hicp()

@app.callback(Output("fig-numbeo","figure"),
              Input("nmb-cat","value"), Input("nmb-mode","value"))
def _nmb(cat,mode): return fig_numbeo(cat,mode)

# WAGES
@app.callback(Output("fig-wages","figure"),
              Input("occ-sel","value"), Input("sal-slider","value"))
def _w(occ,g): return fig_wages(occ,g)

# CITY
@app.callback(Output("fig-urban","figure"), Input("sal-slider","value"))
def _ua(_): return fig_urban()

# WHAT IF?
@app.callback(Output("fig-whatif","figure"),
              Input("sal-slider","value"), Input("fi-offer","value"))
def _wi(gi,gf): return fig_whatif(gi,gf)

# CHOICE STAGE — weight value displays
for cid in ["salary"] + [c[0] for c in CRITERIA]:
    @app.callback(Output(f"wt-val-{cid}","children"),
                  Input(f"wt-{cid}","value"),
                  prevent_initial_call=True)
    def _wv(v): return str(v)

@app.callback(
    Output("fig-choice","figure"),
    Output("verdict-box","children"),
    *[Input(f"wt-{cid}","value") for cid in ["salary"]+[c[0] for c in CRITERIA]],
    Input("sal-slider","value"),
    Input("fi-offer","value"))
def _choice(*args):
    n_crit = 1 + len(CRITERIA)
    wt_vals = list(args[:n_crit])
    gross_it, gross_fi = args[n_crit], args[n_crit+1]
    crit_ids = ["salary"] + [c[0] for c in CRITERIA]
    weights  = dict(zip(crit_ids, wt_vals))
    fig, verdict, vcol = fig_choice(weights, gross_it, gross_fi)
    verdict_el = html.Div([
        html.Strong("Decision verdict: ", style={"color":C["text"]}),
        html.Span(verdict, style={"color":vcol}),
        html.Br(),
        html.Span("Weights: 0 = ignore this factor · 3 = moderate importance · 5 = crucial. "
                  "Line thickness in chart reflects your weight for each criterion.",
                  style={"color":C["muted"],"fontSize":"11px"}),
    ])
    return fig, verdict_el

# DATASETS
@app.callback(Output("fig-estat","figure"), Input("estat-choice","value"))
def _estat(choice): return fig_estat(choice)

# ─── RUN ──────────────────────────────────────────────────────────────────────
def run_app():
    app.run(host='0.0.0.0', port=port, debug=False, use_reloader=False)

threading.Thread(target=run_app, daemon=True).start()



Loading datasets …
  ✓ tableau_salary_range.csv                                 271 rows
  ✓ tableau_breakeven.csv                                    101 rows
  ✓ tableau_hicp_categories.csv                               12 rows
  ✓ tableau_fi_wage_distribution.csv                          60 rows
  ✓ tableau_urban_audit_indicators.csv                        16 rows
  ✓ numbeo_comparison.csv                                     57 rows
  ✓ estat_urb_living_conditions.csv                       174779 rows
  ✓ estat_urb_transport.csv                                78550 rows
  ✓ estat_ilc_di03_income_nuts2.csv                        20934 rows
  ✓ estat_lfst_employment_rates.csv                        12334 rows
  ✓ estat_ilc_lvho01_housing_overburden.csv                 4224 rows
  ✓ estat_ilc_lvho07a_housing_cost_share.csv                6732 rows
  → Using reference Urban Audit values
  ✓ Interaction logging callbacks registered
Dash is running on http://0.0.0.0:45889/



INFO:dash.dash:Dash is running on http://0.0.0.0:45889/



In [4]:
output.serve_kernel_port_as_iframe(port, height = 1000)

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive'

In [19]:
wages2 = load('/content/statfin_wages_by_occupation.csv')
print(wages2['Tiedot_label'].unique())

  ✓ /content/statfin_wages_by_occupation.csv                 108 rows
['Number of full-time wage and salary earners'
 'Average for total earnings of full-time wage and salary earners, EUR per month'
 '1st decile of total earnings of full-time wage and salary earners, EUR per month'
 'Median for total earnings of full-time wage and salary earners, EUR per month'
 '9th decile of total earnings of full-time wage and salary earners, EUR per month'
 'Average for earnings for regular working hours of full-time wage and salary earners, EUR per month'
 '1st decile of earnings for regular working hours of full-time wage and salary earners, EUR per month'
 'Median for earnings for regular working hours of full-time wage and salary earners, EUR per month'
 '9th decile of earnings for regular working hours of full-time wage and salary earners, EUR per month']


In [20]:
print(e_lc.columns.tolist())

['dataset', 'time', 'obs_value', 'freq', 'indic_ur', 'cities']
